# Meta-heurísticas - 2026/2
## Atividade T1

### Gerador de Instâncias



In [14]:
import random
import os

def gerar_instancia_com_intersecao(n_pessoas, nome_arquivo):
    # 1. Definir o tamanho do núcleo (ex: 20% das pessoas, mínimo de 4)
    tamanho_nucleo = max(4, int(n_pessoas * 0.2))
    if tamanho_nucleo % 2 != 0:
        tamanho_nucleo += 1 # Garante que seja par para facilitar a divisão

    pessoas = list(range(1, n_pessoas + 1))
    random.shuffle(pessoas)

    nucleo = pessoas[:tamanho_nucleo]
    restantes = pessoas[tamanho_nucleo:]

    saldos = {p: 0 for p in pessoas}

    # 2. Atribuir saldos ao núcleo para que a soma de todos eles seja ZERO
    soma_nucleo = 0
    for p in nucleo[:-1]:
        valor = random.randint(-500, 500)
        while valor == 0:
            valor = random.randint(-500, 500)
        saldos[p] = valor
        soma_nucleo += valor
    # O último do núcleo absorve a diferença para zerar o grupo
    saldos[nucleo[-1]] = -soma_nucleo 

    # 3. Dividir o núcleo em pares (subgrupos)
    k_grupos = tamanho_nucleo // 2
    subgrupos_nucleo = [nucleo[i:i + 2] for i in range(0, tamanho_nucleo, 2)]

    # 4. Dividir as pessoas "restantes" em 'k_grupos' de forma equilibrada
    subgrupos_restantes = [[] for _ in range(k_grupos)]
    for i, p in enumerate(restantes):
        subgrupos_restantes[i % k_grupos].append(p)

    # 5. Fazer a mágica da interseção: o (subgrupo do nucleo) + (subgrupo restante) deve somar ZERO
    for i in range(k_grupos):
        soma_sub_nucleo = sum(saldos[p] for p in subgrupos_nucleo[i])
        grupo_restante = subgrupos_restantes[i]

        if not grupo_restante:
            continue

        soma_temp_restante = 0
        # Gera saldos aleatórios para quase todos do grupo restante
        for p in grupo_restante[:-1]:
            valor = random.randint(-500, 500)
            while valor == 0:
                valor = random.randint(-500, 500)
            saldos[p] = valor
            soma_temp_restante += valor

        # A peça chave: O último integrante do grupo restante assume a dívida exata 
        # para que a soma do (sub_nucleo + restante) zere.
        saldos[grupo_restante[-1]] = -(soma_sub_nucleo + soma_temp_restante)

    # 6. Remover quem ficou com zero e embaralhar a lista final
    pessoas_info = [{"id": p, "saldo": saldos[p]} for p in pessoas if saldos[p] != 0]
    pessoas_info.sort(key=lambda x: x["id"])

    # 7. Salvar no arquivo
    with open(nome_arquivo, 'w') as f:
        for pessoa in pessoas_info:
            f.write(f"{pessoa['id']} {pessoa['saldo']}\n")
            
    # Salvar metadados para você conseguir analisar a qualidade da sua busca local depois
    os.makedirs("data/metadados", exist_ok=True)
    nome_base = os.path.basename(nome_arquivo)
    nome_arquivo_meta = f"data/metadados/{nome_base.replace('.txt', '_metadados.txt')}"
    
    with open(nome_arquivo_meta, 'w') as f:
        f.write("=== METADADOS (GABARITO DA INSTÂNCIA) ===\n")
        f.write(f"Tamanho Núcleo: {tamanho_nucleo} | Quantidade de Grupos Ótimos: {k_grupos}\n")
        f.write(f"-> O algoritmo Guloso tende a errar gerando {n_pessoas - 2} transações.\n")
        f.write(f"-> O Ótimo Global plantado é de {n_pessoas - k_grupos} transações.\n")
        f.write("=========================================\n")

In [15]:
valores = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400, 420, 440, 460, 480, 500]
random.seed(42)

os.makedirs("data/instancias", exist_ok=True)

for numeros in valores:
    caminho_arquivo = f"data/instancias/instancia_splitwise_{numeros}.txt"
    gerar_instancia_com_intersecao(numeros, caminho_arquivo)

### Leitura dos arquivos

In [18]:
import pandas as pd
import time

def ler_instancia_para_df(nome_arquivo, n_pessoas):
    df = pd.read_csv(nome_arquivo, sep=" ", names=["id", "saldo"])
    return df

In [17]:
instancias = {}
tamanhos = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400, 420, 440, 460, 480, 500]

for tamanho in tamanhos:
    caminho_arquivo = f"data/instancias/instancia_splitwise_{tamanho}.txt"
    df_carregado = ler_instancia_para_df(caminho_arquivo, tamanho)

    instancias[tamanho] = df_carregado

### Iniciando busca de solução

#### Algoritmo Construtivo
*    Guloso



In [ ]:
def construtivo_guloso(df_instancia):
    df_temp = df_instancia.copy()
    inicio_tempo = time.perf_counter()

    credores = df_temp[df_temp['saldo'] > 0].values.tolist()
    devedores = df_temp[df_temp['saldo'] < 0].values.tolist()
    transacoes = []

    while credores and devedores:
        devedores.sort(key=lambda x: abs(x[1]), reverse=True)
        credores.sort(key=lambda x: x[1], reverse=True)

        match_encontrado = False

        mapa_credores = {cred[1]: j for j, cred in enumerate(credores)}

        for i, dev in enumerate(devedores):
            saldo_pendente = abs(dev[1])

            if saldo_pendente in mapa_credores:
                j = mapa_credores[saldo_pendente]
                cred = credores[j]

                transacoes.append({
                    'devedor': int(dev[0]),
                    'credor': int(cred[0]),
                    'valor': cred[1]
                })

                devedores.pop(i)
                credores.pop(j)
                match_encontrado = True
                break

        if not match_encontrado:
            dev = devedores[0]
            cred = credores[0]

            valor_transacao = min(abs(dev[1]), cred[1])

            transacoes.append({
                'devedor': int(dev[0]),
                'credor': int(cred[0]),
                'valor': valor_transacao
            })

            devedores[0][1] += valor_transacao
            credores[0][1] -= valor_transacao

            if devedores[0][1] == 0:
                devedores.pop(0)
            if credores[0][1] == 0:
                credores.pop(0)

    fim_tempo = time.perf_counter()
    tempo_execucao = fim_tempo - inicio_tempo

    return transacoes, tempo_execucao

*    Aleatório


In [ ]:
def construtivo_randomizado(df_instancia, alpha=0.2):
    df_temp = df_instancia.copy()
    inicio_tempo = time.perf_counter()

    credores = df_temp[df_temp['saldo'] > 0].values.tolist()
    devedores = df_temp[df_temp['saldo'] < 0].values.tolist()

    transacoes = []

    while credores and devedores:
        devedores.sort(key=lambda x: x[1])
        credores.sort(key=lambda x: x[1], reverse=True)

        limite_dev = max(1, int(alpha * len(devedores)))
        idx_dev = random.randrange(limite_dev)
        dev = devedores[idx_dev]
        magnitude_dev = -dev[1]

        idx_cred = -1
        for j, cred in enumerate(credores):
            if cred[1] == magnitude_dev:
                idx_cred = j
                break

        if idx_cred == -1:
            limite_cred = max(1, int(alpha * len(credores)))
            idx_cred = random.randrange(limite_cred)

        cred = credores[idx_cred]

        valor_transacao = min(magnitude_dev, cred[1])
        transacoes.append({
            'devedor': int(dev[0]),
            'credor': int(cred[0]),
            'valor': valor_transacao
        })

        devedores[idx_dev][1] += valor_transacao
        credores[idx_cred][1] -= valor_transacao

        if devedores[idx_dev][1] == 0:
            devedores.pop(idx_dev)
        if credores[idx_cred][1] == 0:
            credores.pop(idx_cred)


    fim_tempo = time.perf_counter()
    tempo_execucao = fim_tempo - inicio_tempo

    return transacoes, tempo_execucao

#### Algoritmo Busca Local
*    First-improvement


In [ ]:
import time
import pandas as pd

def aplicar_swap(solucao, i, j):
    """
    Pega duas transações nos índices i e j, cruza os pagamentos e consolida.
    Retorna uma nova lista de transações vizinha.
    """
    t1 = solucao[i]
    t2 = solucao[j]

    # Copia a solução ignorando as duas transações que vamos alterar
    nova_solucao = [t for idx, t in enumerate(solucao) if idx != i and idx != j]

    v1, v2 = t1['valor'], t2['valor']
    delta = min(v1, v2)

    # Se uma transação era maior que a outra, sobra um "troco" que continua na rota original
    if v1 > delta:
        nova_solucao.append({'devedor': t1['devedor'], 'credor': t1['credor'], 'valor': v1 - delta})
    if v2 > delta:
        nova_solucao.append({'devedor': t2['devedor'], 'credor': t2['credor'], 'valor': v2 - delta})

    # Cruza os credores com o valor do delta
    nova_solucao.append({'devedor': t1['devedor'], 'credor': t2['credor'], 'valor': delta})
    nova_solucao.append({'devedor': t2['devedor'], 'credor': t1['credor'], 'valor': delta})

    # CONSOLIDAÇÃO: Junta transações duplicadas que tenham o mesmo devedor e credor
    mapa = {}
    for t in nova_solucao:
        if t['devedor'] == t['credor']:
            continue # Ignora se alguém acabou pagando a si mesmo

        par = (t['devedor'], t['credor'])
        if par not in mapa:
            mapa[par] = 0
        mapa[par] += t['valor']

    # Retorna o novo formato limpo
    return [{'devedor': d, 'credor': c, 'valor': v} for (d, c), v in mapa.items()]

In [ ]:
def aplicar_movimento(transacoes_atuais, transacoes_remover, transacoes_adicionar):
    nova_solucao = transacoes_atuais.copy()

    for t_rem in transacoes_remover:
        if t_rem in nova_solucao:
            nova_solucao.remove(t_rem)

    nova_solucao.extend(transacoes_adicionar)
    return nova_solucao

In [ ]:
def buscar_movimento_ciclo(transacoes_atuais):
    grafo = {}
    for t in transacoes_atuais:
        u = t['devedor']
        if u not in grafo:
            grafo[u] = []
        grafo[u].append((t['credor'], t['valor'], t))

    visitados = set()
    rec_stack = []
    edge_stack = []

    def dfs(u):
        visitados.add(u)
        rec_stack.append(u)

        if u in grafo:
            for v, valor, t_original in grafo[u]:
                edge_stack.append(t_original)

                if v not in visitados:
                    ciclo_encontrado = dfs(v)
                    if ciclo_encontrado:
                        return ciclo_encontrado

                elif v in rec_stack:
                    idx_inicio_ciclo = rec_stack.index(v)
                    transacoes_ciclo = edge_stack[idx_inicio_ciclo:]
                    return transacoes_ciclo

                edge_stack.pop()

        rec_stack.pop()
        return None

    for devedor_inicial in list(grafo.keys()):
        if devedor_inicial not in visitados:
            ciclo = dfs(devedor_inicial)

            if ciclo:
                t_remover = ciclo

                min_valor = min(t['valor'] for t in ciclo)

                t_adicionar = []
                for t in ciclo:
                    novo_valor = t['valor'] - min_valor
                    if novo_valor > 0:
                        t_adicionar.append({
                            'devedor': t['devedor'],
                            'credor': t['credor'],
                            'valor': novo_valor
                        })

                return t_remover, t_adicionar

    return [], []

In [ ]:
def bl_first_improvement(solucao_inicial):
    solucao = solucao_inicial.copy()
    inicio_tempo = time.perf_counter()
    n_it = 0

    while True:
        n_it += 1
        melhoria_encontrada = False
        n = len(solucao)

        for i in range(n):
            for j in range(i + 1, n):
                vizinho = aplicar_swap(solucao, i, j)

                if len(vizinho) < len(solucao):
                    solucao = vizinho
                    melhoria_encontrada = True
                    break
            if melhoria_encontrada:
                break

        if not melhoria_encontrada:
            break

    tempo_total = time.perf_counter() - inicio_tempo
    return solucao, tempo_total, n_it

*    Best-improvement

In [ ]:
def buscar_todos_movimentos_ciclo(transacoes_atuais):
    grafo = {}
    for t in transacoes_atuais:
        u = t['devedor']
        if u not in grafo:
            grafo[u] = []
        grafo[u].append((t['credor'], t['valor'], t))

    movimentos_encontrados = []

    for devedor_inicial in list(grafo.keys()):
        pass

    return movimentos_encontrados

In [ ]:
def bl_best_improvement(solucao_inicial):
    solucao = solucao_inicial.copy()
    inicio_tempo = time.perf_counter()
    n_it = 0

    while True:
        n_it += 1
        n = len(solucao)
        melhor_vizinho = None
        menor_qtd_transacoes = len(solucao)

        for i in range(n):
            for j in range(i + 1, n):
                vizinho = aplicar_swap(solucao, i, j)

                if len(vizinho) < menor_qtd_transacoes:
                    menor_qtd_transacoes = len(vizinho)
                    melhor_vizinho = vizinho

        if melhor_vizinho is not None:
            solucao = melhor_vizinho
        else:
            break

    tempo_total = time.perf_counter() - inicio_tempo
    return solucao, tempo_total, n_it

### RESULTADOS FINAIS

In [ ]:
resultados_tabela = []

for tamanho, df_inst in instancias.items():
    # 1. Executa Algoritmos Construtivos
    transacoes_g, tempo_g = construtivo_guloso(df_inst)
    transacoes_r, tempo_r = construtivo_randomizado(df_inst)

    # 2. Executa First-Improvement sobre as duas soluções iniciais
    # sol_fi_g, t_fi_g, nit_fi_g = bl_first_improvement(transacoes_g)
    # sol_fi_r, t_fi_r, nit_fi_r = bl_first_improvement(transacoes_r)

    # 3. Executa Best-Improvement sobre as duas soluções iniciais
    # sol_bi_g, t_bi_g, nit_bi_g = bl_best_improvement(transacoes_g)
    # sol_bi_r, t_bi_r, nit_bi_r = bl_best_improvement(transacoes_r)

    # 4. Grava os resultados consolidados
    resultados_tabela.append({
        'I': f"Inst_{tamanho}",

        'AC_G (sol)': len(transacoes_g),
        'AC_G (t(s))': round(tempo_g, 6),
        'AC_R (sol)': len(transacoes_r),
        'AC_R (t(s))': round(tempo_r, 6),

        # 'BL_FI(AC_G) (sol)': len(sol_fi_g),
        # 'BL_FI(AC_G) (t(s))': round(t_fi_g, 6),
        # 'BL_FI(AC_G) (N_It)': nit_fi_g,

        # 'BL_FI(AC_R) (sol)': len(sol_fi_r),
        # 'BL_FI(AC_R) (t(s))': round(t_fi_r, 6),
        # 'BL_FI(AC_R) (N_It)': nit_fi_r,

        # 'BL_BI(AC_G) (sol)': len(sol_bi_g),
        # 'BL_BI(AC_G) (t(s))': round(t_bi_g, 6),
        # 'BL_BI(AC_G) (N_It)': nit_bi_g,

        # 'BL_BI(AC_R) (sol)': len(sol_bi_r),
        # 'BL_BI(AC_R) (t(s))': round(t_bi_r, 6),
        # 'BL_BI(AC_R) (N_It)': nit_bi_r
    })

df_resultados_parciais = pd.DataFrame(resultados_tabela)
print(df_resultados_parciais.to_string())

           I  AC_G (sol)  AC_G (t(s))  AC_R (sol)  AC_R (t(s))
0    Inst_10           9     0.001991           9     0.002268
1    Inst_20          13     0.001101          14     0.001059
2    Inst_30          26     0.001246          27     0.001172
3    Inst_40          32     0.001317          33     0.001239
4    Inst_50          38     0.001376          37     0.001532
5    Inst_60          52     0.001860          55     0.001606
6    Inst_70          60     0.001950          62     0.001747
7    Inst_80          62     0.001913          65     0.001704
8    Inst_90          71     0.002209          76     0.001936
9   Inst_100          83     0.002620          88     0.002214
10  Inst_120          92     0.004003         103     0.002559
11  Inst_140         107     0.003715         121     0.003134
12  Inst_160         116     0.004122         130     0.003553
13  Inst_180         128     0.004649         148     0.003424
14  Inst_200         159     0.003845         174     0